# MaskDINO (R50) — Fine-tuning sobre tiras de cómic
**TFG Sergio Salaices — UAH 2026**

Dataset: 320 train / 80 val | 5 clases | Pretrained COCO

Instalación: ~10-15 min la primera vez (compila ops custom de atención deformable)

In [ ]:
# ── Celda 1: Instalar Detectron2 + MaskDINO ───────────────────────────────────
import subprocess, sys, os, torch, warnings

# Suprimir warnings de compatibilidad del propio MaskDINO
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=SyntaxWarning)

MASKDINO_DIR = '/tmp/MaskDINO'
torch_ver  = '.'.join(torch.__version__.split('.')[:2])
cuda_ver   = torch.version.cuda.replace('.', '')
wheel_url  = (f'https://dl.fbaipublicfiles.com/detectron2/wheels'
              f'/cu{cuda_ver}/torch{torch_ver}/index.html')

print(f'PyTorch {torch_ver} | CUDA {cuda_ver}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (!)"}')
print(f'VRAM: {round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1)} GB')

# ── Detectron2 ────────────────────────────────────────────────────────────────
res = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'detectron2', '-f', wheel_url, '-q'],
    capture_output=True, text=True
)
if res.returncode == 0:
    print('✓ Detectron2 instalado desde wheel precompilado')
else:
    print('Wheel no disponible — instalando desde fuente (~15 min)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         'git+https://github.com/facebookresearch/detectron2.git', '-q'], check=True)
    print('✓ Detectron2 instalado desde fuente')

# ── MaskDINO ──────────────────────────────────────────────────────────────────
if not os.path.exists(MASKDINO_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/IDEACVR/MaskDINO.git',
         MASKDINO_DIR, '--depth=1'], check=True)
    print('✓ MaskDINO clonado')
else:
    print('✓ MaskDINO ya clonado')

# Dependencias
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'scipy', 'einops', 'timm',
     'git+https://github.com/cocodataset/panopticapi.git'],
    check=True)

# MaskDINO no tiene setup.py — añadir directamente al path
if MASKDINO_DIR not in sys.path:
    sys.path.insert(0, MASKDINO_DIR)
print('✓ MaskDINO añadido a sys.path')

# Parchear ms_deform_attn_func.py para usar PyTorch puro
# (las ops CUDA de MaskDINO no compilan con PyTorch >= 2.x)
func_file = os.path.join(MASKDINO_DIR,
    'maskdino/modeling/pixel_decoder/ops/functions/ms_deform_attn_func.py')

patched = (
    'from __future__ import absolute_import, print_function, division\n'
    'import torch\n'
    'import torch.nn.functional as F\n'
    '\n'
    'def ms_deform_attn_core_pytorch(value, value_spatial_shapes, sampling_locations, attention_weights):\n'
    '    N_, S_, M_, D_ = value.shape\n'
    '    _, Lq_, M_, L_, P_, _ = sampling_locations.shape\n'
    '    value_list = value.split([H_ * W_ for H_, W_ in value_spatial_shapes], dim=1)\n'
    '    sampling_grids = 2 * sampling_locations - 1\n'
    '    sampling_value_list = []\n'
    '    for lid_, (H_, W_) in enumerate(value_spatial_shapes):\n'
    '        value_l_ = value_list[lid_].flatten(2).transpose(1, 2).reshape(N_*M_, D_, H_, W_)\n'
    '        sampling_grid_l_ = sampling_grids[:, :, :, lid_].transpose(1, 2).flatten(0, 1)\n'
    '        sampling_value_l_ = F.grid_sample(value_l_, sampling_grid_l_,\n'
    '                                          mode=\'bilinear\', padding_mode=\'zeros\', align_corners=False)\n'
    '        sampling_value_list.append(sampling_value_l_)\n'
    '    attention_weights = attention_weights.transpose(1, 2).reshape(N_*M_, 1, Lq_, L_*P_)\n'
    '    output = (torch.stack(sampling_value_list, dim=-2).flatten(-2) * attention_weights).sum(-1).view(N_, M_*D_, Lq_)\n'
    '    return output.transpose(1, 2).contiguous()\n'
    '\n'
    'class MSDeformAttnFunction:\n'
    '    @staticmethod\n'
    '    def apply(value, value_spatial_shapes, value_level_start_index,\n'
    '              sampling_locations, attention_weights, im2col_step):\n'
    '        return ms_deform_attn_core_pytorch(\n'
    '            value, value_spatial_shapes, sampling_locations, attention_weights)\n'
)

with open(func_file, 'w') as fh:
    fh.write(patched)
print('✓ ms_deform_attn_func.py parcheado (PyTorch puro, sin CUDA ops)')

# Verificar que el import funciona antes de continuar
try:
    import maskdino
    print('✓ MaskDINO importable correctamente')
except Exception as e:
    raise RuntimeError(f'MaskDINO no se puede importar tras la instalación: {e}')

print('✓ Instalación completa')

In [ ]:
# ── Celda 2: Imports ──────────────────────────────────────────────────────────
import os, cv2, json, csv, random, glob
import numpy as np
import torch
import matplotlib.pyplot as plt

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.data import (
    DatasetCatalog, MetadataCatalog,
    DatasetMapper, build_detection_train_loader
)
from detectron2.data.datasets import register_coco_instances
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.utils.visualizer import Visualizer, ColorMode
import detectron2.data.transforms as T

# MaskDINO — registra la arquitectura en Detectron2
# La ruta exacta de add_maskdino_config varía según la versión del repo
import maskdino
try:
    from maskdino import add_maskdino_config
except ImportError:
    from maskdino.config import add_maskdino_config

MASKDINO_DIR = '/tmp/MaskDINO'  # redundante pero útil si esta celda se re-ejecuta sola

print('Detectron2:', detectron2.__version__)
print('MaskDINO importado ✓')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (!)')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), 'GB')

In [ ]:
# ── Celda 3: Localizar dataset ────────────────────────────────────────────────
BASE = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'instances_train.json' in files:
        candidate = os.path.dirname(root)
        if os.path.isdir(os.path.join(candidate, 'images')):
            BASE = candidate
            break

if BASE is None:
    print('Contenido de /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth < 7:
            print('  ' * depth + os.path.basename(root) + '/')
    raise RuntimeError('Dataset no encontrado — revisa las rutas arriba')

TRAIN_JSON = f'{BASE}/annotations/instances_train.json'
VAL_JSON   = f'{BASE}/annotations/instances_val.json'
TRAIN_IMGS = f'{BASE}/images/train'
VAL_IMGS   = f'{BASE}/images/val'

with open(TRAIN_JSON) as f:
    meta = json.load(f)
cats = {c['id']: c['name'] for c in meta['categories']}

print(f'Dataset en: {BASE}')
print(f'Train: {len(meta["images"])} imgs | Clases: {cats}')

# try/except es más robusto que "in DatasetCatalog" entre versiones de Detectron2
for split in ('comic_train', 'comic_val'):
    try:
        DatasetCatalog.remove(split)
    except KeyError:
        pass

register_coco_instances('comic_train', {}, TRAIN_JSON, TRAIN_IMGS)
register_coco_instances('comic_val',   {}, VAL_JSON,   VAL_IMGS)
print('Datasets registrados ✓')

In [ ]:
# ── Celda 4: Descargar pesos preentrenados en COCO ────────────────────────────
import os, subprocess, sys

WEIGHTS_DIR  = '/kaggle/working/pretrained'
WEIGHTS_FILE = 'maskdino_r50_50ep_instance.pth'
WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, WEIGHTS_FILE)

# URL correcta: los pesos están en detrex-storage, no en el repo de MaskDINO
WEIGHTS_URL = (
    'https://github.com/IDEA-Research/detrex-storage/releases/download/maskdino-v0.1.0/'
    'maskdino_r50_50ep_300q_hid2048_3sd1_instance_maskenhanced_mask46.3ap_box51.7ap.pth'
)

os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Borrar archivo corrupto si existe pero pesa < 100 MB
if os.path.exists(WEIGHTS_PATH):
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1024**2
    if size_mb < 100:
        print(f'Archivo previo corrupto ({size_mb:.1f} MB) — eliminando...')
        os.remove(WEIGHTS_PATH)

if not os.path.exists(WEIGHTS_PATH):
    print('Descargando pesos MaskDINO R50 (~400 MB)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True)
    import requests
    with requests.get(WEIGHTS_URL, stream=True, allow_redirects=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(WEIGHTS_PATH, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total and downloaded % (20 * 1024 * 1024) < 8192:
                    print(f'  {downloaded/1024**2:.0f} / {total/1024**2:.0f} MB', flush=True)

    size_mb = os.path.getsize(WEIGHTS_PATH) / 1024**2
    if size_mb < 100:
        os.remove(WEIGHTS_PATH)
        raise RuntimeError(f'Descarga incompleta ({size_mb:.1f} MB). Revisa la URL.')
    print(f'✓ Descargado: {size_mb:.0f} MB')
else:
    print(f'✓ Pesos ya descargados: {os.path.getsize(WEIGHTS_PATH)/1024**2:.0f} MB')

print(f'Ruta: {WEIGHTS_PATH}')

In [ ]:
# ── Celda 5: Configuración ────────────────────────────────────────────────────
OUTPUT_DIR  = '/kaggle/working/maskdino_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Buscar el config
CONFIG_FILE = os.path.join(MASKDINO_DIR,
    'configs/coco/instance-segmentation/maskdino_R50_bs16_50ep_3s.yaml')

if not os.path.exists(CONFIG_FILE):
    candidates = (
        glob.glob(os.path.join(MASKDINO_DIR, 'configs/**/*R50*instance*50ep*.yaml'), recursive=True) or
        glob.glob(os.path.join(MASKDINO_DIR, 'configs/**/*R50*instance*.yaml'),      recursive=True) or
        glob.glob(os.path.join(MASKDINO_DIR, 'configs/**/*R50*.yaml'),               recursive=True)
    )
    if not candidates:
        all_cfgs = glob.glob(os.path.join(MASKDINO_DIR, 'configs/**/*.yaml'), recursive=True)
        print('Configs disponibles:')
        for f in all_cfgs:
            print(' ', os.path.relpath(f, MASKDINO_DIR))
        raise RuntimeError('No se encontró ningún config de MaskDINO R50')
    CONFIG_FILE = candidates[0]
    print(f'Config alternativo: {os.path.relpath(CONFIG_FILE, MASKDINO_DIR)}')

cfg = get_cfg()
add_maskdino_config(cfg)

# Permitir claves desconocidas del YAML antes de hacer merge
cfg.set_new_allowed(True)
cfg.merge_from_file(CONFIG_FILE)
cfg.set_new_allowed(False)

# Dataset
cfg.DATASETS.TRAIN = ('comic_train',)
cfg.DATASETS.TEST  = ('comic_val',)
cfg.DATALOADER.NUM_WORKERS = 2

# Pesos preentrenados
cfg.MODEL.WEIGHTS = WEIGHTS_PATH

# 5 clases
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 5

# Congelar primeras 2 etapas del backbone
cfg.MODEL.BACKBONE.FREEZE_AT = 2

# Solver fine-tuning: 4000 iters (~25 épocas sobre 320 imgs)
cfg.SOLVER.IMS_PER_BATCH     = 2
cfg.SOLVER.BASE_LR           = 0.0001
cfg.SOLVER.MAX_ITER          = 4000
cfg.SOLVER.STEPS             = (2500, 3500)
cfg.SOLVER.GAMMA             = 0.1
cfg.SOLVER.WARMUP_ITERS      = 200
cfg.SOLVER.WARMUP_FACTOR     = 1.0 / 1000
cfg.SOLVER.CHECKPOINT_PERIOD = 1000
cfg.SOLVER.WEIGHT_DECAY      = 0.05
# 'norm' es el tipo válido en Detectron2 para clip por norma L2
cfg.SOLVER.CLIP_GRADIENTS.ENABLED    = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE  = 'norm'
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 0.1
cfg.SOLVER.CLIP_GRADIENTS.NORM_TYPE  = 2.0

cfg.TEST.EVAL_PERIOD = 1000
cfg.OUTPUT_DIR = OUTPUT_DIR

# Necesario para que el DatasetMapper cargue gt_masks y las convierta a bitmask
cfg.MODEL.MASK_ON       = True
cfg.INPUT.MASK_FORMAT   = 'bitmask'

print('Configuración MaskDINO lista')
print(f'  Config:    {os.path.basename(CONFIG_FILE)}')
print(f'  CLASSES=   {cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES}')
print(f'  FREEZE_AT= {cfg.MODEL.BACKBONE.FREEZE_AT}')
print(f'  MAX_ITER=  {cfg.SOLVER.MAX_ITER} | LR= {cfg.SOLVER.BASE_LR} | batch= {cfg.SOLVER.IMS_PER_BATCH}')
print(f'  Salida:    {OUTPUT_DIR}')

In [ ]:
# ── Celda 6: Trainer con augmentación ─────────────────────────────────────────
class ComicMaskDINOTrainer(DefaultTrainer):

    @classmethod
    def build_train_loader(cls, cfg):
        base_mapper = DatasetMapper(
            is_train=True,
            augmentations=[
                T.RandomFlip(horizontal=True),
                T.ResizeShortestEdge(
                    short_edge_length=[480, 512, 544, 576, 608, 640],
                    max_size=1000,
                    sample_style='choice',
                ),
                T.RandomBrightness(0.8, 1.2),
                T.RandomContrast(0.8, 1.2),
            ],
            image_format=cfg.INPUT.FORMAT,
            use_instance_mask=True,
            instance_mask_format='bitmask',  # BitMasks en vez de PolygonMasks
        )

        def mapped(dataset_dict):
            d = base_mapper(dataset_dict)
            if 'instances' in d and d['instances'].has('gt_masks'):
                # MaskDINO.prepare_targets espera tensor raw (N,H,W) float,
                # no el objeto BitMasks de Detectron2
                d['instances'].gt_masks = d['instances'].gt_masks.tensor.float()
            return d

        return build_detection_train_loader(cfg, mapper=mapped)

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(dataset_name, output_dir=cfg.OUTPUT_DIR)

print('ComicMaskDINOTrainer definido ✓')

In [ ]:
# ── Celda 7: ENTRENAMIENTO ────────────────────────────────────────────────────
# Si ya existe model_final.pth se salta el entrenamiento
if os.path.exists(os.path.join(OUTPUT_DIR, 'model_final.pth')):
    print('✓ model_final.pth ya existe — se omite el entrenamiento')
else:
    print('Iniciando entrenamiento (~1.5h en T4)...')
    trainer = ComicMaskDINOTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()

In [ ]:
# ── Celda 8: Evaluación final ─────────────────────────────────────────────────
cfg.MODEL.WEIGHTS = os.path.join(OUTPUT_DIR, 'model_final.pth')

predictor  = DefaultPredictor(cfg)
evaluator  = COCOEvaluator('comic_val', output_dir=OUTPUT_DIR)
val_loader = build_detection_test_loader(cfg, 'comic_val')
results    = inference_on_dataset(predictor.model, val_loader, evaluator)

print('\n========== RESULTADOS FINALES MaskDINO ==========')
for task, metrics in results.items():
    print(f'\n[{task}]')
    for k, v in metrics.items():
        print(f'  {k:30s}: {v:.2f}')

In [ ]:
# ── Celda 9: Guardar métricas en CSV ──────────────────────────────────────────
CLASSES = ['Titulo_Periodico', 'Fecha', 'Comic', 'Titulo_Comic', 'Autor']
bbox = results.get('bbox', {})
segm = results.get('segm', {})

row = {
    'modelo'   : 'MaskDINO R50',
    'AP_box'   : round(bbox.get('AP',   0), 2),
    'AP50_box' : round(bbox.get('AP50', 0), 2),
    'AP75_box' : round(bbox.get('AP75', 0), 2),
    'APs_box'  : round(bbox.get('APs',  0), 2),
    'APm_box'  : round(bbox.get('APm',  0), 2),
    'APl_box'  : round(bbox.get('APl',  0), 2),
    'AP_mask'  : round(segm.get('AP',   0), 2),
    'AP50_mask': round(segm.get('AP50', 0), 2),
    'AP75_mask': round(segm.get('AP75', 0), 2),
}
for cls in CLASSES:
    row[f'AP_{cls}_box']  = round(bbox.get(f'AP-{cls}', 0), 2)
    row[f'AP_{cls}_mask'] = round(segm.get(f'AP-{cls}', 0), 2)

csv_path = os.path.join(OUTPUT_DIR, 'metricas_maskdino.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    writer.writeheader()
    writer.writerow(row)

print(f'Métricas guardadas en: {csv_path}')
print(f"  AP box  : {row['AP_box']}")
print(f"  AP50 box: {row['AP50_box']}")
print(f"  AP mask : {row['AP_mask']}")
print(f"  AP50 msk: {row['AP50_mask']}")

In [ ]:
# ── Celda 10: Visualización de predicciones ───────────────────────────────────
metadata  = MetadataCatalog.get('comic_val')
val_dicts = DatasetCatalog.get('comic_val')
samples   = random.sample(val_dicts, min(6, len(val_dicts)))

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
fig.suptitle('MaskDINO R50 — Predicciones sobre val', fontsize=14, fontweight='bold')

for ax, d in zip(axes.flatten(), samples):
    img_bgr = cv2.imread(d['file_name'])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    try:
        outputs   = predictor(img_bgr)
        instances = outputs['instances'].to('cpu')
        v   = Visualizer(img_rgb, metadata=metadata, scale=1.0,
                         instance_mode=ColorMode.SEGMENTATION)
        vis = v.draw_instance_predictions(instances)
        ax.imshow(vis.get_image())
        ax.set_title(f'{os.path.basename(d["file_name"])}  [{len(instances)} det.]', fontsize=8)
    except Exception as e:
        ax.imshow(img_rgb)
        ax.set_title(f'Error: {str(e)[:50]}', fontsize=7, color='red')
    ax.axis('off')

plt.tight_layout()
vis_path = os.path.join(OUTPUT_DIR, 'predicciones_val.png')
plt.savefig(vis_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en: {vis_path}')

In [ ]:
# ── Celda 11: Listado de archivos de salida ───────────────────────────────────
print('Archivos generados:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1024**2
        print(f'  {os.path.relpath(full, OUTPUT_DIR):45s}  {size:7.1f} MB')